# 🎙️ Fish Audio S2 Pro F16 (GGUF) on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/digitaldeveloperinfo/tts-s2-pro-fp16gguf/blob/main/s2_pro_f16_colab.ipynb)

Notebook ini menjalankan model **Fish Audio S2 Pro (F16 GGUF)** menggunakan inference engine native C++/CUDA **[s2.cpp](https://github.com/rodrigomatta/s2.cpp)**.

### 📋 Informasi Model:
- **Model:** `rodrigomt/s2-pro-gguf` (`s2-pro-f16.gguf` ~9.9 GB)
- **Arsitektur:** Dual-AR (~4.56B params; Slow-AR Qwen3 36-layer + Fast-AR 4-layer + RVQ codec)
- **Backend:** GGML CUDA accelerated (NVIDIA T4 / L4 / A100)
- **Catatan VRAM:** File model F16 berukuran ~9.9 GB. Muat di GPU Google Colab standar (NVIDIA T4 15 GB VRAM / L4 24 GB / A100 40 GB).


## 1. Cek GPU & Install Dependensi Sistem

In [ ]:
# 1. Pastikan runtime Colab terhubung ke GPU (T4 / L4 / A100)
!nvidia-smi

# 2. Install dependensi build sistem & utilitas download HF cepat
!apt-get update -qq && apt-get install -y -qq cmake build-essential libsndfile1-dev
!pip install -q -U huggingface_hub hf_transfer


## 2. Clone Repository s2.cpp & Compile dengan CUDA

In [ ]:
import os

# Clone s2.cpp jika belum ada
if not os.path.exists("s2.cpp"):
    !git clone --recurse-submodules https://github.com/rodrigomatta/s2.cpp.git

%cd s2.cpp

# Build dengan backend CUDA
!cmake -B build -DCMAKE_BUILD_TYPE=Release -DS2_CUDA=ON
!cmake --build build --parallel $(nproc)

# Verifikasi binary hasil compile
!ls -la build/s2


## 3. Unduh Model `s2-pro-f16.gguf` dan `tokenizer.json`
Menggunakan `hf_transfer` untuk kecepatan unduh maksimal.

In [ ]:
import os
from huggingface_hub import hf_hub_download

# Aktifkan akselerasi unduhan hf-transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
repo_id = "rodrigomt/s2-pro-gguf"

print("Mengunduh tokenizer.json (~12 MB)...")
hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=".")

print("Mengunduh s2-pro-f16.gguf (~9.9 GB)... Mohon tunggu beberapa saat...")
hf_hub_download(repo_id=repo_id, filename="s2-pro-f16.gguf", local_dir=".")

print("Semua file model berhasil diunduh!")


## 4. Jalankan Sintesis Suara (Text-to-Speech)
Ubah variabel `text_input` sesuai kalimat yang ingin disintesis.

In [ ]:
from IPython.display import Audio, display

text_input = "Halo semuanya! Ini adalah uji coba text-to-speech menggunakan model Fish Audio S2 Pro F16 GGUF di Google Colab."
output_wav = "output_tts.wav"

# Jalankan inferensi dengan s2 CLI (backend CUDA 0)
!./build/s2 \
  --model s2-pro-f16.gguf \
  --tokenizer tokenizer.json \
  --cuda 0 \
  --text "{text_input}" \
  --output "{output_wav}"

# Putar file audio hasil sintesis
display(Audio(output_wav, rate=44100))


## 5. Voice Cloning (Zero-Shot Clone)
Anda dapat meniru gaya dan timbre suara dari file audio referensi (5–30 detik WAV/MP3 mono).

In [ ]:
import os
from IPython.display import Audio, display

# Path ke file audio referensi yang telah Anda upload (misal: 'sample_voice.wav')
ref_audio = "sample_voice.wav"
ref_transcript = "Transkrip kalimat persis yang diucapkan pada audio referensi di atas."
target_text = "Ini adalah kalimat baru yang disuarakan menggunakan suara dari file referensi tadi."
cloned_wav = "cloned_output.wav"

if os.path.exists(ref_audio):
    !./build/s2 \
      --model s2-pro-f16.gguf \
      --tokenizer tokenizer.json \
      --cuda 0 \
      --prompt-audio "{ref_audio}" \
      --prompt-text "{ref_transcript}" \
      --text "{target_text}" \
      --output "{cloned_wav}"

    display(Audio(cloned_wav, rate=44100))
else:
    print(f"File '{ref_audio}' tidak ditemukan.")
    print("Tips: Upload audio pendek (5-30 detik format .wav/.mp3) ke folder s2.cpp di Colab untuk mencoba Voice Cloning!")


## 6. (Opsional) Menjalankan HTTP API Server
`s2.cpp` juga menyediakan mode server HTTP bawaan (port 3030).

In [ ]:
# Menjalankan s2 dalam mode HTTP Server (di background)
import subprocess
import time

server_proc = subprocess.Popen([
    "./build/s2",
    "--model", "s2-pro-f16.gguf",
    "--tokenizer", "tokenizer.json",
    "--cuda", "0",
    "--server",
    "--host", "127.0.0.1",
    "--port", "3030"
])

print("Server sedang berjalan di background (PID:", server_proc.pid, ")...")
time.sleep(5)  # Tunggu server inisialisasi

# Contoh pemanggilan POST /generate via curl
!curl -X POST http://127.0.0.1:3030/generate \
  --form "text=Halo dari HTTP Server s2.cpp!" \
  --form 'params={"max_new_tokens":512,"temperature":0.7}' \
  -o server_output.wav

display(Audio("server_output.wav", rate=44100))

# Untuk mematikan server setelah selesai:
# server_proc.terminate()
